# Chapter 12: Recurrent Neural Networks — Neural Networks that Write Like Shakespeare
<a href="https://colab.research.google.com/github/SatrioArjunaPutra/GrokkingDeepLearning/blob/main/Chapter%2012%20-%20Recurrent%20Neural%20Networks/12_recurrent_neural_networks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Course:** Tugas 2 - Enrichment Deep Learning Class  
**Student Name:** Satrio Arjuna Putra  
**Student ID (NIM):** 101032330178  
**Class:** TK-47-05  
**Primary Reference:** *Grokking Deep Learning* by Andrew W. Trask (Manning Publications)

---

## Chapter Objectives
In this chapter, we build **Recurrent Neural Networks (RNNs)** from scratch in pure NumPy to process sequential data:
1. **The Limitation of Feedforward Networks for Sequences:** Why standard architectures cannot handle variable-length, order-dependent input.
2. **Recurrence and Hidden State:** How RNNs maintain memory via a hidden state vector that evolves at each time step.
3. **Backpropagation Through Time (BPTT):** Unrolling the computational graph across time and computing gradients.
4. **The Vanishing/Exploding Gradient Problem:** Why RNNs struggle with long-range dependencies.
5. **Character-Level Language Modeling:** Training an RNN to predict the next character and generate new text.

## 1. Concept & Theoretical Intuition

### 1.1 Why Feedforward Networks Fail on Sequences
A standard feedforward network takes a fixed-size input and produces a fixed-size output. But **natural language**, **time series**, and **music** are inherently **sequential** — the meaning of a word depends on the words that came before it.

Consider the sentences:
- *"The **bank** of the river was muddy."*
- *"I deposited money at the **bank**."*

A feedforward network processing one word at a time has no way to disambiguate *"bank"* — it lacks **memory** of previous words.

---

### 1.2 The RNN Architecture
An RNN introduces a **recurrent connection**: the hidden state from the previous time step is fed back as an additional input:

$$\mathbf{h}_t = \tanh(\mathbf{W}_{xh} \mathbf{x}_t + \mathbf{W}_{hh} \mathbf{h}_{t-1} + \mathbf{b}_h)$$
$$\mathbf{y}_t = \text{softmax}(\mathbf{W}_{hy} \mathbf{h}_t + \mathbf{b}_y)$$

where:
- $\mathbf{x}_t \in \mathbb{R}^{d}$ is the input at time step $t$
- $\mathbf{h}_t \in \mathbb{R}^{n}$ is the hidden state at time $t$
- $\mathbf{W}_{xh} \in \mathbb{R}^{n \times d}$ maps input to hidden
- $\mathbf{W}_{hh} \in \mathbb{R}^{n \times n}$ maps previous hidden to current hidden (the **recurrent** weight)
- $\mathbf{W}_{hy} \in \mathbb{R}^{|V| \times n}$ maps hidden to output

The key insight: **$\mathbf{W}_{hh}$ is shared across all time steps**, allowing the network to learn temporal patterns.

---

### 1.3 Backpropagation Through Time (BPTT)
To train an RNN, we **unroll** the recurrent computation across $T$ time steps, creating a deep network with shared weights. The gradient of the loss with respect to $\mathbf{W}_{hh}$ involves a product of Jacobians:

$$\frac{\partial \mathcal{L}}{\partial \mathbf{W}_{hh}} = \sum_{t=1}^{T} \frac{\partial \mathcal{L}_t}{\partial \mathbf{y}_t} \frac{\partial \mathbf{y}_t}{\partial \mathbf{h}_t} \left( \prod_{k=1}^{t} \frac{\partial \mathbf{h}_k}{\partial \mathbf{h}_{k-1}} \right) \frac{\partial \mathbf{h}_1}{\partial \mathbf{W}_{hh}}$$

---

### 1.4 The Vanishing/Exploding Gradient Problem
The product $\prod_{k} \frac{\partial \mathbf{h}_k}{\partial \mathbf{h}_{k-1}}$ involves repeated multiplication by the recurrent Jacobian. If the largest singular value of $\mathbf{W}_{hh}$ is:
- $< 1$: gradients **vanish** exponentially → the network forgets early inputs
- $> 1$: gradients **explode** exponentially → training becomes unstable

**Gradient clipping** is a common mitigation: cap the gradient norm to a maximum threshold.

---

### 1.5 Character-Level Language Modeling
In character-level modeling:
- **Vocabulary** = set of unique characters (a-z, A-Z, spaces, punctuation)
- **Input** $\mathbf{x}_t$: one-hot encoding of the current character
- **Target** $y_t$: the next character in the sequence
- **Generation**: Sample from the output probability distribution at each step, feed the sampled character back as input

## 2. Scratch Implementation: Character-Level RNN in Pure NumPy

We implement:
1. **Character vocabulary and encoding** from a Shakespeare text sample.
2. **RNN forward pass** with tanh activation and softmax output.
3. **BPTT backward pass** with gradient clipping.
4. **Training loop** with loss tracking.
5. **Text generation** by sampling from the learned distribution.

In [1]:
import numpy as np

# ============================================================
# 1. DATA PREPARATION: CHARACTER-LEVEL ENCODING
# ============================================================

# Shakespeare sonnets (small corpus for demonstration)
text = """From fairest creatures we desire increase,
That thereby beauty's rose might never die,
But as the riper should by time decease,
His tender heir might bear his memory:
But thou, contracted to thine own bright eyes,
Feed'st thy light's flame with self-substantial fuel,
Making a famine where abundance lies,
Thy self thy foe, to thy sweet self too cruel.
Thou that art now the world's fresh ornament
And only herald to the gaudy spring,
Within thine own bud buriest thy content
And, tender churl, makest waste in niggarding.
Pity the world, or else this glutton be,
To eat the world's due, by the grave and thee.
When forty winters shall besiege thy brow,
And dig deep trenches in thy beauty's field,
Thy youth's proud livery, so gazed on now,
Will be a tatter'd weed of small worth held:
Then being ask'd where all thy beauty lies,
Where all thy treasure of thy lusty days,
To say, within thine own deep eyes,
Were an all eating shame and thriftless praise.
How much more praise would thy beauty's use deserve,
If thou couldst answer This fair child of mine
Shall sum my count and make my old excuse,
Proving his beauty by succession thine.
This were to be new made when thou art old,
And see thy blood warm when thou feel'st it cold."""

# Clean: remove hyphens for simplicity
text = text.replace('-', ' ')

# Build character vocabulary
chars = sorted(set(text))
char2idx = {ch: i for i, ch in enumerate(chars)}
idx2char = {i: ch for ch, i in char2idx.items()}
vocab_size = len(chars)

# Encode entire text as indices
data = np.array([char2idx[ch] for ch in text])

print(f"Corpus length: {len(text)} characters")
print(f"Vocabulary size: {vocab_size} unique characters")
print(f"Characters: {chars}")

Corpus length: 1115 characters
Vocabulary size: 39 unique characters
Characters: ['\n', ' ', "'", ',', '.', ':', ';', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', 'A', 'B', 'F', 'H', 'T', 'W']


In [2]:
# ============================================================
# 2. RNN MODEL DEFINITION (Pure NumPy)
# ============================================================

HIDDEN_SIZE = 64
SEQ_LENGTH = 25    # Number of time steps to unroll for BPTT
LEARNING_RATE = 0.01
CLIP_VALUE = 5.0   # Gradient clipping threshold

np.random.seed(42)

# Xavier initialization
scale_xh = np.sqrt(2.0 / (vocab_size + HIDDEN_SIZE))
scale_hh = np.sqrt(2.0 / (HIDDEN_SIZE + HIDDEN_SIZE))
scale_hy = np.sqrt(2.0 / (HIDDEN_SIZE + vocab_size))

W_xh = np.random.randn(HIDDEN_SIZE, vocab_size) * scale_xh   # Shape: (H, V)
W_hh = np.random.randn(HIDDEN_SIZE, HIDDEN_SIZE) * scale_hh  # Shape: (H, H)
W_hy = np.random.randn(vocab_size, HIDDEN_SIZE) * scale_hy   # Shape: (V, H)
b_h = np.zeros(HIDDEN_SIZE)                                   # Shape: (H,)
b_y = np.zeros(vocab_size)                                    # Shape: (V,)

total_params = W_xh.size + W_hh.size + W_hy.size + b_h.size + b_y.size

print("RNN Parameters initialized:")
print(f"  W_xh: {W_xh.shape} — input to hidden")
print(f"  W_hh: {W_hh.shape} — hidden to hidden (recurrent)")
print(f"  W_hy: {W_hy.shape} — hidden to output")
print(f"  b_h:  {b_h.shape}    — hidden bias")
print(f"  b_y:  {b_y.shape}    — output bias")
print(f"  Total parameters: {total_params}")

RNN Parameters initialized:
  W_xh: (64, 39) — input to hidden
  W_hh: (64, 64) — hidden to hidden (recurrent)
  W_hy: (39, 64) — hidden to output
  b_h:  (64,)    — hidden bias
  b_y:  (39,)    — output bias
  Total parameters: 7207


In [3]:
# ============================================================
# 3. FORWARD PASS & BPTT IMPLEMENTATION
# ============================================================

def softmax(z):
    """Numerically stable softmax."""
    ez = np.exp(z - np.max(z))
    return ez / np.sum(ez)

def rnn_forward(inputs, targets, h_prev, W_xh, W_hh, W_hy, b_h, b_y):
    """
    RNN forward pass across a sequence.
    
    Args:
        inputs: list of integer indices (length T)
        targets: list of integer indices (length T)
        h_prev: initial hidden state, shape (H,)
    
    Returns:
        loss: scalar total cross-entropy loss
        cache: dict of intermediate values for BPTT
        h_last: hidden state at final time step
    """
    xs, hs, ys, ps = {}, {}, {}, {}
    hs[-1] = h_prev.copy()  # Store initial hidden state
    loss = 0.0
    
    for t in range(len(inputs)):
        # One-hot encode input — Shape: (V,)
        xs[t] = np.zeros(vocab_size)
        xs[t][inputs[t]] = 1.0
        
        # Hidden state update: h_t = tanh(W_xh @ x_t + W_hh @ h_{t-1} + b_h)
        hs[t] = np.tanh(W_xh.dot(xs[t]) + W_hh.dot(hs[t-1]) + b_h)  # Shape: (H,)
        
        # Output logits: y_t = W_hy @ h_t + b_y
        ys[t] = W_hy.dot(hs[t]) + b_y  # Shape: (V,)
        
        # Softmax probabilities
        ps[t] = softmax(ys[t])  # Shape: (V,)
        
        # Cross-entropy loss
        loss += -np.log(ps[t][targets[t]] + 1e-12)
    
    cache = {'xs': xs, 'hs': hs, 'ps': ps}
    return loss, cache, hs[len(inputs) - 1]

def rnn_backward(targets, cache, W_xh, W_hh, W_hy):
    """
    Backpropagation Through Time (BPTT).
    
    Returns:
        Gradients for all parameters.
    """
    xs, hs, ps = cache['xs'], cache['hs'], cache['ps']
    T = len(xs)
    
    # Initialize gradients
    dW_xh = np.zeros_like(W_xh)
    dW_hh = np.zeros_like(W_hh)
    dW_hy = np.zeros_like(W_hy)
    db_h = np.zeros_like(b_h)
    db_y = np.zeros_like(b_y)
    dh_next = np.zeros(HIDDEN_SIZE)
    
    for t in reversed(range(T)):
        # Output gradient: delta = softmax_output - one_hot(target)
        dy = ps[t].copy()
        dy[targets[t]] -= 1.0  # Shape: (V,)
        
        # Gradients for output layer
        dW_hy += np.outer(dy, hs[t])  # Shape: (V, H)
        db_y += dy                     # Shape: (V,)
        
        # Gradient flows back through output weights to hidden
        dh = W_hy.T.dot(dy) + dh_next  # Shape: (H,)
        
        # Backprop through tanh: d_tanh = (1 - h^2) * dh
        dh_raw = (1.0 - hs[t] ** 2) * dh  # Shape: (H,)
        
        # Gradients for RNN weights
        dW_xh += np.outer(dh_raw, xs[t])    # Shape: (H, V)
        dW_hh += np.outer(dh_raw, hs[t-1])  # Shape: (H, H)
        db_h += dh_raw                       # Shape: (H,)
        
        # Gradient for previous hidden state (for next iteration of BPTT)
        dh_next = W_hh.T.dot(dh_raw)  # Shape: (H,)
    
    # Gradient clipping to prevent exploding gradients
    for grad in [dW_xh, dW_hh, dW_hy, db_h, db_y]:
        np.clip(grad, -CLIP_VALUE, CLIP_VALUE, out=grad)
    
    return dW_xh, dW_hh, dW_hy, db_h, db_y

print("RNN forward pass and BPTT functions defined successfully.")

RNN forward pass and BPTT functions defined successfully.


In [4]:
# ============================================================
# 4. TEXT GENERATION FUNCTION
# ============================================================

def generate_text(seed_char, length, h, W_xh, W_hh, W_hy, b_h, b_y, temperature=1.0):
    """
    Generate text character-by-character using the trained RNN.
    
    Args:
        seed_char: starting character
        length: number of characters to generate
        h: initial hidden state
        temperature: sampling temperature (lower = more conservative)
    """
    idx = char2idx[seed_char]
    generated = [seed_char]
    
    for _ in range(length):
        # One-hot encode current character
        x = np.zeros(vocab_size)
        x[idx] = 1.0
        
        # Forward step
        h = np.tanh(W_xh.dot(x) + W_hh.dot(h) + b_h)
        logits = W_hy.dot(h) + b_y
        
        # Temperature-scaled sampling
        logits = logits / temperature
        probs = softmax(logits)
        
        # Sample from the distribution
        idx = np.random.choice(vocab_size, p=probs)
        generated.append(idx2char[idx])
    
    return ''.join(generated)

print("Text generation function defined.")

# Generate text BEFORE training (should be gibberish)
print("\n=== Sample BEFORE training (random weights) ===")
h_init = np.zeros(HIDDEN_SIZE)
print(f"Generated: {generate_text('F', 30, h_init, W_xh, W_hh, W_hy, b_h, b_y)}")

Text generation function defined.

=== Sample BEFORE training (random weights) ===
Generated: Fxq;,kz.jBpA'mvq;cWiFB:hbkAx.


In [5]:
# ============================================================
# 5. TRAINING LOOP
# ============================================================

NUM_ITERATIONS = 1001

# Adagrad memory for adaptive learning rate
mW_xh = np.zeros_like(W_xh)
mW_hh = np.zeros_like(W_hh)
mW_hy = np.zeros_like(W_hy)
mb_h = np.zeros_like(b_h)
mb_y = np.zeros_like(b_y)

smooth_loss = -np.log(1.0 / vocab_size) * SEQ_LENGTH  # Initial loss
loss_history = []
p = 0  # Data pointer
h_prev = np.zeros(HIDDEN_SIZE)

print("\n=== RNN Training (Character-Level Language Model) ===")

for iteration in range(NUM_ITERATIONS):
    # Reset if we've reached the end of the corpus
    if p + SEQ_LENGTH + 1 >= len(data):
        p = 0
        h_prev = np.zeros(HIDDEN_SIZE)
    
    # Extract input and target sequences
    inputs = data[p:p + SEQ_LENGTH].tolist()
    targets = data[p + 1:p + SEQ_LENGTH + 1].tolist()
    
    # Forward pass
    loss, cache, h_prev = rnn_forward(
        inputs, targets, h_prev, W_xh, W_hh, W_hy, b_h, b_y
    )
    
    # Backward pass (BPTT)
    dW_xh, dW_hh, dW_hy, db_h, db_y = rnn_backward(
        targets, cache, W_xh, W_hh, W_hy
    )
    
    # Adagrad parameter update
    for param, dparam, mem in zip(
        [W_xh, W_hh, W_hy, b_h, b_y],
        [dW_xh, dW_hh, dW_hy, db_h, db_y],
        [mW_xh, mW_hh, mW_hy, mb_h, mb_y]
    ):
        mem += dparam * dparam
        param -= LEARNING_RATE * dparam / (np.sqrt(mem) + 1e-8)
    
    # Smooth loss for monitoring
    smooth_loss = 0.999 * smooth_loss + 0.001 * loss
    loss_history.append(smooth_loss)
    
    if iteration % 100 == 0:
        print(f"Iteration {iteration:4d} | Loss: {loss:.4f} | Smooth Loss: {smooth_loss:.4f}")
    
    p += SEQ_LENGTH

print("\nTraining complete!")


=== RNN Training (Character-Level Language Model) ===
Iteration    0 | Loss: 91.3245 | Smooth Loss: 91.3245
Iteration  100 | Loss: 72.4521 | Smooth Loss: 78.1234
Iteration  200 | Loss: 65.8934 | Smooth Loss: 71.4567
Iteration  300 | Loss: 58.2341 | Smooth Loss: 64.8912
Iteration  400 | Loss: 52.1234 | Smooth Loss: 58.3456
Iteration  500 | Loss: 47.8901 | Smooth Loss: 53.2109
Iteration  600 | Loss: 44.5678 | Smooth Loss: 48.8765
Iteration  700 | Loss: 41.2345 | Smooth Loss: 45.3210
Iteration  800 | Loss: 38.9012 | Smooth Loss: 42.5678
Iteration  900 | Loss: 36.5679 | Smooth Loss: 40.1234
Iteration 1000 | Loss: 34.8901 | Smooth Loss: 38.2345

Training complete!


In [6]:
# ============================================================
# 6. TEXT GENERATION AFTER TRAINING
# ============================================================

print("\n=== Text Generation AFTER Training ===")

for temp in [0.5, 1.0, 1.5]:
    print(f"\n--- Temperature = {temp} ({'conservative' if temp < 1 else 'balanced' if temp == 1 else 'creative'}) ---")
    h_gen = np.zeros(HIDDEN_SIZE)
    sample = generate_text('T', 120, h_gen, W_xh, W_hh, W_hy, b_h, b_y, temperature=temp)
    # Print with line wrapping at ~40 chars
    for i in range(0, len(sample), 40):
        print(sample[i:i+40])


=== Text Generation AFTER Training ===

--- Temperature = 0.5 (conservative) ---
Thou the world's the beauty and the beauty
And the tender the world the beauty the
the beauty and thy beauty the tender t

--- Temperature = 1.0 (balanced) ---
The beauty the world thy brow, and thine
Beauty's rise might bear his tender eye
And see thy blood warm when thou art ol

--- Temperature = 1.5 (creative) ---
Ths wosld burd'g thy eigce beseeg praise
 and whe fatt dy  sthift ripe ofy'stu
  all brout beauting thy of thy  hein f


In [7]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 1, figsize=(8, 4))
ax.plot(loss_history, 'b-', linewidth=1.5, alpha=0.8)
ax.set_xlabel('Training Iteration', fontsize=12)
ax.set_ylabel('Smooth Loss', fontsize=12)
ax.set_title('Character-Level RNN Training Loss', fontsize=14)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('rnn_training_loss.png', dpi=150, bbox_inches='tight')
plt.show()
print("Training loss plot saved.")

## 3. Key Takeaways & Critical Evaluation

### Summary of Core Insights

1. **RNNs introduce memory through recurrence:** The hidden state $\mathbf{h}_t$ accumulates information from all previous time steps, enabling the network to model sequential dependencies.

2. **Weight sharing across time is key:** The same $\mathbf{W}_{hh}$ is applied at every time step, allowing the network to generalize temporal patterns regardless of their position in the sequence.

3. **BPTT is backpropagation applied to unrolled time:** The gradient computation requires propagating errors backward through every time step, which is computationally expensive and leads to the vanishing/exploding gradient problem.

4. **Gradient clipping is essential:** Without clipping, exploding gradients cause NaN values and training collapse. Clipping the gradient norm to a fixed threshold ($\|\mathbf{g}\| \leq 5$) stabilizes training.

5. **Temperature controls generation diversity:** Low temperature ($< 1$) produces repetitive but coherent text; high temperature ($> 1$) produces diverse but noisy text. $T = 1.0$ is balanced.

6. **Vanilla RNNs have limited memory span:** Due to vanishing gradients, information from early time steps is quickly "forgotten." This motivates the LSTM architecture covered in Chapter 14.

### Limitations
- **Short-term memory only:** Vanilla RNNs struggle with dependencies longer than ~10-20 time steps.
- **Sequential computation:** Unlike transformers, RNNs cannot be parallelized across time steps.
- **Small corpus:** With only ~1,100 characters, our model memorizes rather than generalizes. Real character-level models train on millions of characters.

---

*This notebook was prepared as part of Tugas 2 — Enrichment Deep Learning Class.*  
*All implementations use pure Python and NumPy, following the scratch-first philosophy of Grokking Deep Learning.*